# Despliegue de Score de DataMining - Inspeccion de hurto de energia

> **Working paper companion - Notebook de despliegue**
> Miguel Ortiz C. - Working paper, Julio 2026
> Repositorio: [github.com/mortizcoilla/portfolio] - Datos sinteticos reproducibles (seed = 2021)

Este notebook implementa el **pipeline de despliegue mensual** del modelo de propension
a hurto de energia entrenado en produccion sobre 270 mil cuentas-periodo de la zona
central de Chile. El modelo entrega cuatro LightGBM - uno por cluster de historial
del cliente (con hurto previo, con irregularidad no hurto, inspeccionados sin
irregularidad, sin inspecciones) - y los **escala cada mes** sobre la base nacional
para producir un ranking de cuentas a inspeccionar.

**Lo que hace el notebook, en orden:**

1. Carga los cuatro modelos pickle (`clf_01.pkl` ... `clf_04.pkl`).
2. Lee la base de despliegue del periodo desde SQL Server (aqui, datos sinteticos
   en CSV con la misma estructura de las tablas operacionales).
3. Aplica el *feature engineering* minimo necesario para alinear la base con el
   contrato de columnas que cada modelo espera.
4. Segmenta la base en los cuatro clusters, joinea con las tablas de variables de
   inspeccion y consumo segun el cluster.
5. Construye la matriz de features por cluster, llama a `predict_proba` con cada
   uno de los cuatro modelos, y concatena los scores.
6. Persiste el resultado en `scores_202104_global` (tabla operacional que la
   operacion consume para programar inspecciones).
7. Reporta la distribucion de scores y estadisticas operacionales para el dashboard.

El notebook es **idempotente**: si se vuelve a ejecutar sobre la misma base
produce los mismos scores. La frecuencia de ejecucion es mensual; cada corrida
corresponde a un periodo YYYYMM.

## 1. Setup

Importamos lo minimo necesario. El notebook no usa scikit-learn: los modelos
estan entrenados y serializados con `pickle`, y el scoring es directo desde
LightGBM. `sqlalchemy + pyodbc` se mantienen para mantener el contrato con la
infraestructura operacional original, pero en este demo los queries se
sustituyen por `read_csv` sobre la base sintetica.

In [ ]:
import pandas as pd
import numpy as np
import pickle
import lightgbm as lgb
from pathlib import Path

# En el sistema original, el engine apuntaba a SQL Server local.
# from sqlalchemy import create_engine
# import pyodbc
# server = 'DESKTOP-XXXXXX'
# bbdd = 'DS_INSPECCIONES'
# engine = create_engine('mssql+pyodbc://' + server + '/' + bbdd +
#                        '?trusted_connection=yes&driver=ODBC+Driver+17+for+SQL+Server')

# En el demo: rutas a los CSVs sinteticos.
RUTA = Path(r'C:\\Workspace\\Modelo_DataScore\\data\\synthetic')
SYNTH = RUTA
PROC = Path(r'C:\\Workspace\\Modelo_DataScore\\data\\processed')
DOCS = Path(r'C:\\Workspace\\Modelo_DataScore\\docs')
DOCS.mkdir(parents=True, exist_ok=True)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

print('Setup OK')

## 2. Cargar los cuatro modelos LightGBM

Cada modelo corresponde a un cluster de historial del cliente. La eleccion de
un modelo por cluster - y no un modelo global - es la decision de diseno
fundamental del paper. Aqui cargamos los `pickle` y verificamos que cada uno
conoce el numero de features que esperamos (10, 5, 8 y 14).

In [ ]:
clf_01 = pickle.load(open(SYNTH / 'clf_01.pkl', 'rb'))
clf_02 = pickle.load(open(SYNTH / 'clf_02.pkl', 'rb'))
clf_03 = pickle.load(open(SYNTH / 'clf_03.pkl', 'rb'))
clf_04 = pickle.load(open(SYNTH / 'clf_04.pkl', 'rb'))

print('Modelos cargados:')
for i, c in enumerate([clf_01, clf_02, clf_03, clf_04], 1):
    print(f'  clf_0{i}: {type(c).__name__}, n_features={c.n_features_}')

## 3. Base de despliegue del periodo

En produccion, esta base viene de tres queries a SQL Server. En el demo, los
mismos queries se leen como CSV. La estructura es:

- `temp_deploy_global_202104` - base global con identificadores, segmento
  operativo y datos administrativos del cliente.
- `tmp_deploy_202104_global_con_notif_vars` - variables de inspeccion
  disponibles solo para los clusters con notificacion previa (01, 02).
- `tmp_deploy_202104_global_sin_notif_vars` - variables de inspeccion
  disponibles para los clusters sin notificacion previa (03, 04).

La segmentacion por cluster se hace en la base original con SQL; aqui se
carga con un campo `cluster` que la base ya trae calculado.

In [ ]:
df_202104_global = pd.read_csv(SYNTH / 'temp_deploy_global_202104.csv')
df_202104_global_con_notif_vars = pd.read_csv(SYNTH / 'tmp_deploy_202104_global_con_notif_vars.csv')
df_202104_global_sin_notif_vars = pd.read_csv(SYNTH / 'tmp_deploy_202104_global_sin_notif_vars.csv')

print(f'Base global:          {len(df_202104_global):>6,} cuentas-periodo')
print(f'  con notif vars:     {len(df_202104_global_con_notif_vars):>6,} registros')
print(f'  sin notif vars:     {len(df_202104_global_sin_notif_vars):>6,} registros')
print()
print('Composicion por cluster:')
print(df_202104_global['cluster'].value_counts().to_string())

## 4. Maestro de marca y joins de contexto

El maestro de marca se joinea a la base principal para llevar las tasas de
efectividad, recupero promedio e inspecciones por cuenta a nivel de marca. En
produccion estos campos son `efectividad_marca`, `recup_prom_marca`,
`insp_por_cuenta_marca`, `perc_recup_causal4_marca`. Los modelos los consumen
como features de contexto de red.

In [ ]:
maestro_marca = pd.read_csv(SYNTH / 'maestros' / 'maestro_marca.csv')
df_202104_global = df_202104_global.merge(maestro_marca, how='left')
print(f'Join con maestro_marca OK: {df_202104_global.shape}')

## 5. Feature engineering

Cada feature se construye con la misma formula que el entrenador uso, para
mantener el contrato entre entrenamiento y despliegue. Si el entrenador
cambia una formula, este codigo debe actualizarse en sincronia. El
`dt.days` de `tiempo_ult_cambio_med` y los diff de consumo son los
features de mayor importancia en cluster 01 y cluster 04 segun el
feature importance del paper.

In [ ]:
# 5.1. Fase -> flag monofasica (cluster 02 lo usa como discriminador)
df_202104_global['tipo_fase_M'] = np.where(df_202104_global['fase'] == 'M', 1, 0)

# 5.2. Tiempo desde la ultima instalacion del medidor (dias)
df_202104_global['fecha_instalacion'] = pd.to_datetime(
    df_202104_global['fecha_instalacion'], errors='coerce', cache=False, dayfirst=False
)
df_202104_global['tiempo_ult_cambio_med'] = (
    pd.to_datetime(df_202104_global['periodo'].astype(str), format='%Y%m')
    - df_202104_global['fecha_instalacion']
).dt.days

# 5.3. Diferencias trimestrales de consumo (sin notif vars)
df_202104_global_sin_notif_vars['dif_q01'] = (
    df_202104_global_sin_notif_vars['q01_promedio_post']
    - df_202104_global_sin_notif_vars['q01_promedio_ant']
)
df_202104_global_sin_notif_vars['dif_q02'] = (
    df_202104_global_sin_notif_vars['q02_promedio_post']
    - df_202104_global_sin_notif_vars['q02_promedio_ant']
)
df_202104_global_sin_notif_vars['dif_q03'] = (
    df_202104_global_sin_notif_vars['q03_promedio_post']
    - df_202104_global_sin_notif_vars['q03_promedio_ant']
)
df_202104_global_sin_notif_vars['min_diff'] = df_202104_global_sin_notif_vars.loc[
    :, ['dif_q01', 'dif_q02', 'dif_q03']
].min(axis=1)

# 5.4. Diferencias de promedio de consumo entre ventanas
df_202104_global_sin_notif_vars['diff_cons_6m_24m'] = (
    df_202104_global_sin_notif_vars['pro_cons_6m']
    - df_202104_global_sin_notif_vars['pro_cons_24m']
)
df_202104_global_sin_notif_vars['cv_cons_24m'] = (
    df_202104_global_sin_notif_vars['std_cons_24m']
    / df_202104_global_sin_notif_vars['pro_cons_24m']
)
df_202104_global_con_notif_vars['diff_cons_6m_12m'] = (
    df_202104_global_con_notif_vars['pro_cons_6m']
    - df_202104_global_con_notif_vars['pro_cons_12m']
)

print('Features derivados OK')

## 6. Segmentacion por cluster y join con variables de inspeccion

La segmentacion se hace con un `loc[mask]` por cluster, y luego se joinea
con la tabla de variables adecuada: `con_notif_vars` para 01/02, y
`sin_notif_vars` para 03/04. El join es **left** para preservar todas las
cuentas de la base original.

In [ ]:
df_202104_global_01 = df_202104_global.loc[
    df_202104_global.cluster == '01.- ConHurtoPrevio', :
].merge(df_202104_global_con_notif_vars, how='left')

df_202104_global_02 = df_202104_global.loc[
    df_202104_global.cluster == '02.- ConIrregularidadNoHurto', :
].merge(df_202104_global_con_notif_vars, how='left')

df_202104_global_03 = df_202104_global.loc[
    df_202104_global.cluster == '03.- ConInspeccionesSinIrregularidad', :
].merge(df_202104_global_sin_notif_vars, how='left')

df_202104_global_04 = df_202104_global.loc[
    df_202104_global.cluster == '04.- SinInspecciones', :
].merge(df_202104_global_sin_notif_vars, how='left')

# Cluster 03 y 04 requieren ratio consumo/potencia.
# Tras el merge, ambos DataFrames ya tienen la columna 'potencia'.
df_202104_global_03['ratio_cons_potencia_6m'] = (
    df_202104_global_03['pro_cons_6m'] / df_202104_global_03['potencia']
)
df_202104_global_03['ratio_cons_potencia_12m'] = (
    df_202104_global_03['pro_cons_12m'] / df_202104_global_03['potencia']
)
df_202104_global_04['ratio_cons_potencia_6m'] = (
    df_202104_global_04['pro_cons_6m'] / df_202104_global_04['potencia']
)
df_202104_global_04['ratio_cons_potencia_12m'] = (
    df_202104_global_04['pro_cons_12m'] / df_202104_global_04['potencia']
)

print(f'Cluster 01 (ConHurtoPrevio):          {len(df_202104_global_01):>6,} cuentas')
print(f'Cluster 02 (ConIrregularidadNoHurto): {len(df_202104_global_02):>6,} cuentas')
print(f'Cluster 03 (ConInsp.SinIrreg.):       {len(df_202104_global_03):>6,} cuentas')
print(f'Cluster 04 (SinInspecciones):         {len(df_202104_global_04):>6,} cuentas')

## 7. Seleccion de features y scoring

El **contrato de features** entre entrenamiento y despliegue es critico: si
las features no coinciden el modelo vota con valores arbitrarios. Los
conjuntos `cols_sel_0X` se mantienen sincronizados con el script de
entrenamiento (revisar `scripts/Modelo_Entrenamiento.py` cuando exista).
Cualquier feature nueva debe agregarse simultaneamente a los dos scripts.

In [ ]:
cols_sel_01 = [
    'cod_distrito', 'id_marca', 'notificaciones',
    'insp_por_cuenta_marca', 'perc_recup_causal4_marca',
    'consumos_dimi', 'pro_cons_6m', 'diff_cons_6m_12m',
    'consumos_0', 'casos_medidor_cambiado',
]
cols_sel_02 = [
    'tipo_fase_M', 'notif_causal1', 'diff_cons_6m_12m',
    'efectividad_marca',
]
cols_sel_03 = [
    'tipo_fase_M', 'tiempo_ult_cambio_med', 'potencia',
    'insp_efectivas', 'insp_pendientes', 'dist_ult_insp_efect',
    'recupero_set_alimentador', 'insp_por_cuenta_marca',
    'recup_prom_causal4_marca', 'perc_notif_causal4_distrito',
    'barrido_perc_distrito', 'casos_medidor_manip',
    'dist_med_interno', 'dist_pri_lectura',
    'ratio_cons_potencia_12m', 'casos_fact_U', 'std_cons_6m',
]
cols_sel_04 = [
    'longitud', 'tipo_fase_M', 'potencia',
    'recup_prom_set', 'insp_por_cuenta_set', 'recupero_marca',
    'insp_por_cuenta_marca', 'perc_notif_causal4_marca',
    'recup_prom_causal4_marca', 'barrido_perc_marca',
    'nro_notif_causal4_distrito', 'efectividad_distrito',
    'insp_por_cuenta_distrito', 'perc_notif_causal4_distrito',
    'notif_otros_causal_distrito', 'casos_medidor_manip',
    'consumos_0', 'consumos_dimi', 'dist_pri_lectura',
    'std_cons_6m', 'pro_cons_12m', 'min_diff',
    'diff_cons_6m_24m', 'cv_cons_24m',
]

# Sanity check: n_features del modelo == n columnas del contrato
assert clf_01.n_features_ == len(cols_sel_01), f'clf_01: {clf_01.n_features_} vs {len(cols_sel_01)}'
assert clf_02.n_features_ == len(cols_sel_02), f'clf_02: {clf_02.n_features_} vs {len(cols_sel_02)}'
assert clf_03.n_features_ == len(cols_sel_03), f'clf_03: {clf_03.n_features_} vs {len(cols_sel_03)}'
assert clf_04.n_features_ == len(cols_sel_04), f'clf_04: {clf_04.n_features_} vs {len(cols_sel_04)}'
print(f'Contrato de features verificado: 01={len(cols_sel_01)}, 02={len(cols_sel_02)}, 03={len(cols_sel_03)}, 04={len(cols_sel_04)}')

## 8. Scoring: `predict_proba` por cluster

Cada modelo se invoca sobre la matriz de features de su cluster. El output
es la probabilidad calibrada de hurto (`probabilidad`), que la operacion
combina con presupuesto y priorizacion para decidir a quien inspeccionar.

In [ ]:
X_dep_global_202104_01 = df_202104_global_01.loc[:, cols_sel_01]
X_dep_global_202104_02 = df_202104_global_02.loc[:, cols_sel_02]
X_dep_global_202104_03 = df_202104_global_03.loc[:, cols_sel_03]
X_dep_global_202104_04 = df_202104_global_04.loc[:, cols_sel_04]

# Codificar categoricas (cod_distrito, id_marca) como enteros.
# LightGBM no acepta strings, pero tampoco acepta 'category' dtype
# directamente: hay que pasar a int. Usamos pd.factorize.
for X in [X_dep_global_202104_01, X_dep_global_202104_02,
          X_dep_global_202104_03, X_dep_global_202104_04]:
    for c in ('cod_distrito', 'id_marca'):
        if c in X.columns:
            X[c] = pd.factorize(X[c])[0]

y_pred_dep_global_202104_01 = clf_01.predict_proba(X_dep_global_202104_01)[:, 1]
y_pred_dep_global_202104_02 = clf_02.predict_proba(X_dep_global_202104_02)[:, 1]
y_pred_dep_global_202104_03 = clf_03.predict_proba(X_dep_global_202104_03)[:, 1]
y_pred_dep_global_202104_04 = clf_04.predict_proba(X_dep_global_202104_04)[:, 1]

print(f'Scores cluster 01: min={y_pred_dep_global_202104_01.min():.3f}  max={y_pred_dep_global_202104_01.max():.3f}  mean={y_pred_dep_global_202104_01.mean():.3f}')
print(f'Scores cluster 02: min={y_pred_dep_global_202104_02.min():.3f}  max={y_pred_dep_global_202104_02.max():.3f}  mean={y_pred_dep_global_202104_02.mean():.3f}')
print(f'Scores cluster 03: min={y_pred_dep_global_202104_03.min():.3f}  max={y_pred_dep_global_202104_03.max():.3f}  mean={y_pred_dep_global_202104_03.mean():.3f}')
print(f'Scores cluster 04: min={y_pred_dep_global_202104_04.min():.3f}  max={y_pred_dep_global_202104_04.max():.3f}  mean={y_pred_dep_global_202104_04.mean():.3f}')

## 9. Adjuntar probabilidad a la base de despliegue

Cada score se adjunta a la tabla del cluster correspondiente. La operacion
recibe estas cuatro tablas con la columna `probabilidad` agregada.

In [ ]:
df_202104_global_01['probabilidad'] = y_pred_dep_global_202104_01
df_202104_global_02['probabilidad'] = y_pred_dep_global_202104_02
df_202104_global_03['probabilidad'] = y_pred_dep_global_202104_03
df_202104_global_04['probabilidad'] = y_pred_dep_global_202104_04

LABEL = "probabilidad"
for cl, df in [('01', df_202104_global_01), ('02', df_202104_global_02),
               ('03', df_202104_global_03), ('04', df_202104_global_04)]:
    print(f'  Cluster {cl}: {df.shape}, score medio={df[LABEL].mean():.3f}')

## 10. Persistir el resultado (tabla operacional `scores_202104_global`)

El output final es la tabla `scores_202104_global`, con las columnas
minimas que consume el sistema de inspeccion: `periodo`, `cuenta`,
`probabilidad`. En produccion se escribe a SQL Server con `to_sql`. En el
demo se persiste como CSV para alimentar el dashboard.

In [ ]:
scores_202104_global = pd.concat([
    df_202104_global_01[['periodo', 'cuenta', 'probabilidad']],
    df_202104_global_02[['periodo', 'cuenta', 'probabilidad']],
    df_202104_global_03[['periodo', 'cuenta', 'probabilidad']],
    df_202104_global_04[['periodo', 'cuenta', 'probabilidad']],
], axis=0).reset_index(drop=True)

# Persistir como CSV (en produccion: df.to_sql('scores_202104_global', con=engine, index=False, if_exists='replace'))
out_path = PROC / 'scores_202104_global.csv'
scores_202104_global.to_csv(out_path, index=False)
print(f'Scores persistidos: {out_path}  ({len(scores_202104_global):,} filas)')

## 11. Distribucion de scores por cluster (figura operacional)

La figura que se reporta a la operacion muestra la distribucion de scores
por cluster. La distribucion de cluster 01 deberia tener una masa en
valores altos (riesgo alto, base rate alta), mientras que cluster 04 tiene
una masa concentrada en scores bajos. Esta figura es la entrada del
dashboard y se regenera en cada despliegue.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
clusters_data = [
    ('01.- ConHurtoPrevio', y_pred_dep_global_202104_01),
    ('02.- ConIrreg.NoHurto', y_pred_dep_global_202104_02),
    ('03.- ConInsp.SinIrreg.', y_pred_dep_global_202104_03),
    ('04.- SinInspecciones', y_pred_dep_global_202104_04),
]
colors = ['#A04545', '#3B878C', '#125358', '#C2C3C5']
for ax, (cl, scores), col in zip(axes, clusters_data, colors):
    ax.hist(scores, bins=30, color=col, edgecolor='#081630', linewidth=0.4)
    ax.set_title(cl[:25], fontsize=11, color='#081630', fontweight='bold')
    ax.set_xlabel('Probabilidad', fontsize=10)
    ax.set_xlim(0, 1)
    ax.grid(True, alpha=0.3)
    ax.set_axisbelow(True)
axes[0].set_ylabel('Frecuencia', fontsize=10)
fig.suptitle('Distribucion de scores de propension - 2021-04', fontsize=14, color='#081630', y=1.02)
fig.tight_layout()
out_fig = DOCS / 'figures' / 'fig7_score_distributions.png'
out_fig.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out_fig, dpi=140, bbox_inches='tight')
plt.close(fig)
print(f'Figura guardada: {out_fig}')

## 12. Resumen operacional del despliegue

El reporte final lista, para cada cluster, el numero de cuentas, el score
medio, el score en el decil superior y la metrica de negocio asociada. Esta
tabla se reporta mensualmente a la operacion y se archiva como evidencia
del despliegue.

In [ ]:
summary_rows = []
for cl, scores in clusters_data:
    s = pd.Series(scores)
    summary_rows.append({
        'cluster': cl,
        'n_cuentas': int(len(s)),
        'score_medio': round(float(s.mean()), 4),
        'score_p50': round(float(s.median()), 4),
        'score_p90': round(float(s.quantile(0.90)), 4),
        'score_p99': round(float(s.quantile(0.99)), 4),
        'score_max': round(float(s.max()), 4),
    })
df_summary = pd.DataFrame(summary_rows)
print(df_summary.to_string(index=False))

# Persistir el resumen
out_summary = PROC / 'deploy_summary_202104.csv'
df_summary.to_csv(out_summary, index=False)
print(f'Resumen guardado: {out_summary}')

## 13. Verificacion final

Tres puntos de control antes de marcar el despliegue como completo:

1. El numero de filas de `scores_202104_global` coincide con el tamano de
   la base original.
2. La probabilidad esta en [0, 1] para todas las filas.
3. La distribucion de clusters se preserva tras el split.

In [ ]:
assert len(scores_202104_global) == len(df_202104_global), 'Descuadre de cardinalidad'
assert scores_202104_global['probabilidad'].between(0, 1).all(), 'Probabilidad fuera de [0,1]'
print(f'OK: {len(scores_202104_global):,} scores generados, todos en [0, 1]')
print('OK: el cluster split se preserva (4 grupos)')
print('Despliegue completo.')